###**Importing Libraries And Creating A Synthetic sales Dataset**

In [1]:
import pandas as pd
import random

random.seed(42)

# Define data parameters
regions = ['North', 'South', 'East', 'West']
categories = ['Electronics', 'Clothing', 'Home & Garden', 'Sports', 'Books']
salespersons = ['Alice', 'Bob', 'Carol', 'David', 'Emma', 'Frank']

# Generate 200 sales transactions
data = {
    'transaction_id': range(1001, 1201),
    'region': [random.choice(regions) for _ in range(200)],
    'category': [random.choice(categories) for _ in range(200)],
    'salesperson': [random.choice(salespersons) for _ in range(200)],
    'sales_amount': [round(random.uniform(50, 5000), 2) for _ in range(200)],
    'customer_id': [random.randint(5000, 5100) for _ in range(200)]
}

df = pd.DataFrame(data)
print(df.head(10))
print(f"\nDataset shape: {df.shape}")


   transaction_id region     category salesperson  sales_amount  customer_id
0            1001  North  Electronics        Emma       4000.80         5075
1            1002  North        Books       Alice       3634.70         5084
2            1003   East  Electronics       David       4210.50         5025
3            1004  South  Electronics        Emma       4601.73         5054
4            1005  South     Clothing         Bob       4904.57         5014
5            1006  South     Clothing       Carol       2693.91         5069
6            1007  North       Sports       Alice       4539.30         5028
7            1008  North       Sports       Frank       2979.87         5082
8            1009   West       Sports       David       3331.85         5019
9            1010  North     Clothing       Alice        465.54         5034

Dataset shape: (200, 6)


###**Task 1: Basic Grouping and Single Aggregations**

In [2]:
print("\n===== Task 1 =====\n")

# Calculate total sales amount for each region using groupby() and sum()
total_sales_per_region = df.groupby('region').agg(total_sales=('sales_amount','sum')).sort_values(by='total_sales', ascending=False).reset_index()
print(f"Total Sales Amount Per Region: -\n{total_sales_per_region}")

# Count the number of transactions for each product category using groupby() and count()
total_transaction_per_category = df.groupby('category').agg(transaction = ('transaction_id','count')).reset_index()
print(f"\nTotal Number Of Transaction Per Category: -\n{total_transaction_per_category}")

# Calculate the average sales amount per salesperson using groupby() and mean()
avg_sales_per_salesperson = df.groupby('salesperson').agg(average_sales = ('sales_amount','mean')).reset_index()
print(f"\nAverage Sales Per Sales Person: -\n{avg_sales_per_salesperson}")


===== Task 1 =====

Total Sales Amount Per Region: -
  region  total_sales
0  South    158977.36
1  North    135181.16
2   West    109383.07
3   East     95189.81

Total Number Of Transaction Per Category: -
        category  transaction
0          Books           39
1       Clothing           42
2    Electronics           45
3  Home & Garden           43
4         Sports           31

Average Sales Per Sales Person: -
  salesperson  average_sales
0       Alice    1998.432273
1         Bob    2554.919063
2       Carol    2454.368571
3       David    2743.036444
4        Emma    2493.457273
5       Frank    2464.135750


### From the above analysis:

The South region has the highest total sales compared to the other regions.

The number of transactions in each category shows which product categories are sold more frequently.

The average sales value helps understand how each salesperson performs per transaction.

###**Task 2: Multi-Column Grouping and Multiple Aggregations**

In [3]:
print("\n===== Task 2 =====\n")

# 1) Group by both region AND category to calculate total sales for each combination
total_sales_region_category = df.groupby(['region','category']).agg(both_total_sales = ('sales_amount','sum')).reset_index()
print(f"Total Sales Per Region And Category: -\n{total_sales_region_category}")

"""
2)
For each salesperson, calculate three metrics simultaneously using the agg() method:
Total sales ('sum')
Average sales ('mean')
Number of transactions ('count')
"""
salesperson_performance = df.groupby('salesperson')['sales_amount'].agg(['sum', 'mean', 'count']).reset_index()
salesperson_performance.columns = ['salesperson','total_sales','average_sales','transaction_count']

# 3) Sort the salesperson results by total sales in descending order to identify the top performer
salesperson_performance = salesperson_performance.sort_values(by='total_sales', ascending=False)
print(f"\nSalesperson Performance: -\n{salesperson_performance}")

# 4) Use .idxmax() on the grouped category sales to find which category has the maximum total revenue
total_revenue = df.groupby('category')['sales_amount'].sum()
highest_revenue_category = total_revenue.idxmax()
print(f"\nMaximum Total Revenue By Category : {highest_revenue_category}")


===== Task 2 =====

Total Sales Per Region And Category: -
   region       category  both_total_sales
0    East          Books          20027.53
1    East       Clothing          19926.20
2    East    Electronics          22791.55
3    East  Home & Garden          15949.08
4    East         Sports          16495.45
5   North          Books          42592.53
6   North       Clothing          18959.06
7   North    Electronics          38889.84
8   North  Home & Garden          19344.48
9   North         Sports          15395.25
10  South          Books          21007.68
11  South       Clothing          50878.36
12  South    Electronics          39229.63
13  South  Home & Garden          22592.00
14  South         Sports          25269.69
15   West          Books          20359.41
16   West       Clothing          16548.81
17   West    Electronics          13553.27
18   West  Home & Garden          33069.36
19   West         Sports          25852.22

Salesperson Performance: -
  salespe

### From the above analysis

In this task, the dataset was grouped by region and category to calculate total sales for each combination.

Salesperson performance was analyzed by calculating total sales, average sales, and transaction count.

The results were sorted to identify the top-performing salesperson.

Finally, the data was grouped by category, and idxmax() was used to find the category with the highest total revenue.

###**Task 3: Custom Aggregation and Complete Sales Report**

In [4]:
print("\n===== Task 3 =====\n")

# 1) Define a custom aggregation function that calculates the sales range (max - min) for each group
def sales_range(x):
    return x.max() - x.min()

# 2) Apply this custom function along with standard aggregations to analyze sales by region
region_sales_analysis = df.groupby('region')['sales_amount'].agg(['sum', 'mean', 'min', 'max', sales_range]).reset_index()
print("Sales Analysis by Region: -\n", region_sales_analysis)

"""
# 3) Create a final summary report that shows for each region:

Total number of transactions (using customer_id with count)
Total sales amount
Average transaction value
 """
summary_report = df.groupby('region').agg({'customer_id':'count', 'sales_amount': ['sum','mean']}).reset_index()
summary_report.columns = ['region', 'customer_count', 'total_sales', 'average_sales']
print(f"\nSummary Report: -\n")
summary_report


===== Task 3 =====

Sales Analysis by Region: -
   region        sum         mean     min      max  sales_range
0   East   95189.81  2213.716512  307.82  4758.46      4450.64
1  North  135181.16  2413.949286  110.71  4788.56      4677.85
2  South  158977.36  2789.076491  373.45  4983.33      4609.88
3   West  109383.07  2485.978864  203.60  4903.53      4699.93

Summary Report: -



,region,customer_count,total_sales,average_sales
0,East,43,95189.81,2213.716512
1,North,56,135181.16,2413.949286
2,South,57,158977.36,2789.076491
3,West,44,109383.07,2485.978864


### From the above analysis

In this task, a custom function sales_range() was created to calculate the difference between the maximum and minimum sales in each region.

The data was grouped by region to compute total sales, average sales, minimum sales, maximum sales, and the sales range.

The columns were renamed to keep the report simple and easier to read.

A final summary report was then generated using dictionary-based aggregation to calculate the number of transactions, total sales, and average sales for each region.

4) Explain the multi-level column structure that results from the dictionary-based aggregation and how it differs from single aggregations

Dictionary aggregation can create multi-level column names, where one level represents the original column and the other represents the aggregation function.


Single aggregations usually produce only one column level because only one function is applied.